# 08-практикалық жұмыс — `train_test_split` командасы

**Пәні:** Желілік қауіптерді анықтауға арналған машиналық оқыту  
**Жұмыс ортасы:** Google Colab / Jupyter Notebook  
**Студент:** Quantqan Shapagat  
**Тобы:** 24-ПО-103-3к-5  

**Мақсаты:** 40 желілік жазбаны талдау, `X` пен `y` құру, `train_test_split` арқылы оқыту/тест жиынына бөлу, сынып қатынасын, индекстерді және деректердің ағып кетпеуін тексеру, салыстыру диаграммаларын жасау және нәтижені CSV/ZIP ретінде сақтау.

Дәптерде PDF-тағы **18 практикалық қадам**, **A–E бекіту жаттығулары**, **мини-сынақ**, **СОӨЖ A/B/C нұсқалары**, **күрделі функция**, **төрт қорытынды** және **15 бақылау сұрағының жауаптары** орындалды. Барлық код ұяшықтарын ретімен орындауға болады.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
from pathlib import Path
from IPython.display import display
from sklearn.model_selection import train_test_split

plt.rcParams['font.family'] = 'DejaVu Sans'
print('pandas:', pd.__version__)
print('train_test_split жұмысына дайын!')

In [ ]:
rng = np.random.default_rng(42)
n_rows = 40
labels = np.array(['normal'] * 24 + ['suspicious'] * 16)

df = pd.DataFrame({
    'session_id': [f'S{i:03d}' for i in range(1, n_rows + 1)],
    'duration_sec': rng.integers(1, 61, n_rows),
    'bytes_sent': rng.integers(100, 11001, n_rows),
    'failed_logins': np.concatenate([
        rng.integers(0, 3, 24),
        rng.integers(2, 10, 16)
    ]),
    'connections_1min': np.concatenate([
        rng.integers(4, 61, 24),
        rng.integers(70, 201, 16)
    ]),
    'label': labels
})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv('network_split_data.csv', index=False)
print('Файл жасалды:', df.shape)
display(df.head())
assert df.shape == (40, 6)

In [ ]:
print('Өлшем:', df.shape)
print('\nБос мәндер:')
print(df.isna().sum())
print('\nҚайталанған жолдар:', df.duplicated().sum())
print('\nДерек түрлері:')
print(df.dtypes)
print('\nСыныптар:')
print(df['label'].value_counts())
assert df.isna().sum().sum() == 0
assert df.duplicated().sum() == 0

In [ ]:
class_counts = df['label'].value_counts()
class_percent = df['label'].value_counts(normalize=True) * 100
print('Сынып саны:')
print(class_counts)
print('\nСынып пайызы:')
print(class_percent.round(1))

In [ ]:
feature_columns = ['duration_sec', 'bytes_sent', 'failed_logins', 'connections_1min']
X = df[feature_columns].copy()
y = df['label'].copy()
print('X өлшемі:', X.shape)
print('y өлшемі:', y.shape)
print('X бағандары:', X.columns.tolist())
print('y атауы:', y.name)
for forbidden_column in ['session_id', 'label']:
    print(forbidden_column, 'X ішінде бар ма?', forbidden_column in X.columns)
assert X.shape == (40, 4)
assert y.shape == (40,)
assert not {'session_id', 'label'}.intersection(X.columns)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)
print('Жалпы жол саны:', len(X_train) + len(X_test))
assert X_train.shape == (30, 4)
assert X_test.shape == (10, 4)
assert y_train.shape == (30,)
assert y_test.shape == (10,)

In [ ]:
print('X_train алғашқы жолдары:'); display(X_train.head())
print('y_train сәйкес мәндері:'); display(y_train.head())
print('X_test алғашқы жолдары:'); display(X_test.head())
print('y_test сәйкес мәндері:'); display(y_test.head())

X_train алғашқы жолдары:


NameError: name 'X_train' is not defined

In [ ]:
def show_class_distribution(name, target):
    counts = target.value_counts().sort_index()
    percent = (target.value_counts(normalize=True).sort_index() * 100).round(1)
    result = pd.DataFrame({'count': counts, 'percent': percent})
    print(f'\n{name}')
    display(result)
    return result

full_distribution = show_class_distribution('Толық дерек', y)
train_distribution = show_class_distribution('Train', y_train)
test_distribution = show_class_distribution('Test', y_test)
assert (train_distribution['percent'] == full_distribution['percent']).all()
assert (test_distribution['percent'] == full_distribution['percent']).all()

In [ ]:
train_indices = set(X_train.index)
test_indices = set(X_test.index)
overlap = train_indices.intersection(test_indices)
print('Train индекс саны:', len(train_indices))
print('Test индекс саны:', len(test_indices))
print('Ортақ индекстер:', overlap)
print('Ортақ индекс саны:', len(overlap))
assert len(overlap) == 0

In [ ]:
split_42_a = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
split_42_b = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
same_test_indices = set(split_42_a[1].index) == set(split_42_b[1].index)
print('42 және 42 бірдей ме?', same_test_indices)
assert same_test_indices
split_7 = train_test_split(X, y, test_size=0.25, random_state=7, stratify=y)
same_42_and_7 = set(split_42_a[1].index) == set(split_7[1].index)
print('42 және 7 бірдей ме?', same_42_and_7)
print('random_state=42 test индекстері:', sorted(split_42_a[1].index))
print('random_state=7 test индекстері:', sorted(split_7[1].index))

In [ ]:
test_sizes = [0.20, 0.25, 0.30]
comparison = []
for size in test_sizes:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=size, random_state=42, stratify=y
    )
    comparison.append({
        'test_size': size,
        'train_rows': len(X_tr),
        'test_rows': len(X_te),
        'train_suspicious': int((y_tr == 'suspicious').sum()),
        'test_suspicious': int((y_te == 'suspicious').sum())
    })
comparison_df = pd.DataFrame(comparison)
display(comparison_df)
comparison_df.to_csv('split_comparison.csv', index=False)

In [ ]:
x_positions = np.arange(len(comparison_df))
bar_width = 0.36
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x_positions - bar_width/2, comparison_df['train_rows'], width=bar_width, label='Train', color='royalblue')
ax.bar(x_positions + bar_width/2, comparison_df['test_rows'], width=bar_width, label='Test', color='darkorange')
ax.set_xticks(x_positions, comparison_df['test_size'].astype(str))
ax.set_title('test_size бойынша train және test өлшемдері')
ax.set_xlabel('test_size')
ax.set_ylabel('Жол саны')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('split_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
_, _, y_train_no_strat, y_test_no_strat = train_test_split(
    X, y, test_size=0.25, random_state=5
)
_, _, y_train_strat, y_test_strat = train_test_split(
    X, y, test_size=0.25, random_state=5, stratify=y
)
print('Stratify жоқ test:')
print(y_test_no_strat.value_counts(normalize=True).round(2))
print('\nStratify бар test:')
print(y_test_strat.value_counts(normalize=True).round(2))

In [ ]:
train_data = df.loc[X_train.index].copy().sort_index().reset_index(drop=True)
test_data = df.loc[X_test.index].copy().sort_index().reset_index(drop=True)
train_data.to_csv('train_data.csv', index=False)
test_data.to_csv('test_data.csv', index=False)
print('train_data.csv:', train_data.shape)
print('test_data.csv:', test_data.shape)
train_check = pd.read_csv('train_data.csv')
test_check = pd.read_csv('test_data.csv')
print('Қайта ашылған train:', train_check.shape)
print('Қайта ашылған test:', test_check.shape)
shared_sessions = set(train_check['session_id']) & set(test_check['session_id'])
print('Ортақ session_id:', shared_sessions)
assert train_check.shape == (30, 6)
assert test_check.shape == (10, 6)
assert not shared_sessions
result_files = ['network_split_data.csv', 'train_data.csv', 'test_data.csv', 'split_comparison.png']
with zipfile.ZipFile('ML_08_results.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file_name in result_files:
        archive.write(file_name)
print('Архив дайын: ML_08_results.zip')

In [ ]:
exercise_a = pd.DataFrame({
    'Барлық жол': [100, 80, 60, 200, 40],
    'test_size': [0.20, 0.25, 0.30, 0.10, 10],
    'Train жолы': [80, 60, 42, 180, 30],
    'Test жолы': [20, 20, 18, 20, 10]
})
display(exercise_a)
assert (exercise_a['Train жолы'] + exercise_a['Test жолы'] == exercise_a['Барлық жол']).all()

In [ ]:
from sklearn.model_selection import train_test_split
X_b = df[['duration_sec', 'bytes_sent', 'failed_logins', 'connections_1min']]
y_b = df['label']
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_b, y_b, test_size=0.25, random_state=42, stratify=y_b
)
print(X_train_b.shape)
print(y_test_b.shape)

In [ ]:
# 1: label X ішінде болмауы керек
X_c = df.drop(columns=['session_id', 'label'])
y_c = df['label']
# 2: нәтижелердің дұрыс реті
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)
# 3: test_size 0 мен 1 арасында немесе дұрыс бүтін жол саны
split_c3 = train_test_split(X_c, y_c, test_size=0.25, random_state=42)
# 4: stratify=y, stratify=X емес
split_c4 = train_test_split(X_c, y_c, test_size=0.20, random_state=42, stratify=y_c)
# 5: бөлек CSV атауы (негізгі train_data.csv, test_data.csv дайын)
train_data.to_csv('exercise_C_train.csv', index=False)
test_data.to_csv('exercise_C_test.csv', index=False)
# 6: индекстер жиыны салыстырылады, values емес
overlap_c = set(X_train_c.index) & set(X_test_c.index)
print('Барлық 6 қате түзетілді. Ортақ индекс:', len(overlap_c))
assert len(overlap_c) == 0

In [ ]:
# 1 импорт, 2 X, 3 y, 4 X бағандары, 5 y сыныптары
from sklearn.model_selection import train_test_split
X_d = df[feature_columns]
y_d = df['label']
print('X бағандары:', X_d.columns.tolist())
print('Сынып саны:', y_d.value_counts().to_dict())
# 6: test_size=20%, 7: random_state=10, 8: stratify=y
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_d, y_d, test_size=0.20, random_state=10, stratify=y_d
)
# 9: төрт shape; 10–11: сынып пайызы; 12: қиылысу; 13: test индекстері
print('Өлшемдер:', X_tr_d.shape, X_te_d.shape, y_tr_d.shape, y_te_d.shape)
print('Train пайыздары:', (y_tr_d.value_counts(normalize=True)*100).round(1).to_dict())
print('Test пайыздары:', (y_te_d.value_counts(normalize=True)*100).round(1).to_dict())
print('Ортақ индекстер:', set(X_tr_d.index) & set(X_te_d.index))
print('Test индекстері:', sorted(X_te_d.index.tolist()))
# 14: train CSV, 15: қайта ашып тексеру
X_tr_d.assign(label=y_tr_d).to_csv('exercise_D_train.csv', index=False)
print('Қайта оқылған CSV өлшемі:', pd.read_csv('exercise_D_train.csv').shape)

In [ ]:
e_42a = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
e_42b = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
e_99 = train_test_split(X, y, test_size=0.25, random_state=99, stratify=y)
e_20 = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
proof = {
    '1. random_state=42 екі рет бірдей': set(e_42a[1].index) == set(e_42b[1].index),
    '2. random_state=42 және 99 әртүрлі': set(e_42a[1].index) != set(e_99[1].index),
    '3. 20% test = 8 жол': len(e_20[1]) == 8,
    '4. 25% stratify кезінде test үлесі 60/40': (e_42a[3].value_counts(normalize=True).sort_index() == y.value_counts(normalize=True).sort_index()).all(),
    '5. Train/test ортақ индекс жоқ': len(set(e_42a[0].index) & set(e_42a[1].index)) == 0,
}
for statement, result in proof.items():
    print(statement, '=>', bool(result))
assert all(proof.values())

NameError: name 'train_test_split' is not defined

In [ ]:
X_mini = df[feature_columns].copy()
y_mini = df['label'].copy()
X_train_mini, X_test_mini, y_train_mini, y_test_mini = train_test_split(
    X_mini, y_mini, test_size=0.30, random_state=21, stratify=y_mini
)
print('Өлшемдер:', X_train_mini.shape, X_test_mini.shape, y_train_mini.shape, y_test_mini.shape)
print('Train сыныптары:', y_train_mini.value_counts().to_dict())
print('Test сыныптары:', y_test_mini.value_counts().to_dict())
print('Ортақ индекстер:', set(X_train_mini.index) & set(X_test_mini.index))
df.loc[X_train_mini.index].to_csv('mini_train.csv', index=False)
df.loc[X_test_mini.index].to_csv('mini_test.csv', index=False)
assert not (set(X_train_mini.index) & set(X_test_mini.index))

In [ ]:
strategies = {'A': (0.20, 10), 'B': (0.25, 20), 'C': (0.30, 30)}
strategy_rows = []
strategy_splits = {}
for name, (size, seed) in strategies.items():
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=size, random_state=seed, stratify=y)
    overlap = len(set(X_tr.index) & set(X_te.index))
    assert overlap == 0
    strategy_splits[name] = (X_tr, X_te, y_tr, y_te)
    train_counts = y_tr.value_counts()
    test_counts = y_te.value_counts()
    strategy_rows.append({
        'Нұсқа': name,
        'test_size': size,
        'random_state': seed,
        'Train жолы': len(X_tr),
        'Test жолы': len(X_te),
        'Train normal': int(train_counts.get('normal', 0)),
        'Train suspicious': int(train_counts.get('suspicious', 0)),
        'Test normal': int(test_counts.get('normal', 0)),
        'Test suspicious': int(test_counts.get('suspicious', 0)),
        'Train normal %': round(float((y_tr == 'normal').mean()*100), 1),
        'Train suspicious %': round(float((y_tr == 'suspicious').mean()*100), 1),
        'Test normal %': round(float((y_te == 'normal').mean()*100), 1),
        'Test suspicious %': round(float((y_te == 'suspicious').mean()*100), 1),
        'Ортақ индекс саны': overlap,
        'Test индекстері': ', '.join(map(str, sorted(X_te.index.tolist())))
    })
    df.loc[X_tr.index].sort_index().to_csv(f'SOOZh_{name}_train.csv', index=False)
    df.loc[X_te.index].sort_index().to_csv(f'SOOZh_{name}_test.csv', index=False)

strategy_df = pd.DataFrame(strategy_rows)
display(strategy_df)
strategy_df.to_csv('SOOZh_comparison.csv', index=False, encoding='utf-8-sig')
for name in strategies:
    print(f'{name}: Test индекстері:', strategy_df.loc[strategy_df['Нұсқа'] == name, 'Test индекстері'].iloc[0])

NameError: name 'train_test_split' is not defined

In [ ]:
positions = np.arange(len(strategy_df))
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(positions - 0.2, strategy_df['Train жолы'], 0.4, label='Train', color='royalblue')
ax.bar(positions + 0.2, strategy_df['Test жолы'], 0.4, label='Test', color='darkorange')
ax.set_xticks(positions, [f'{row["Нұсқа"]} ({row["test_size"]:.2f})' for _, row in strategy_df.iterrows()])
ax.set_title('СОӨЖ: үш бөлу стратегиясының өлшемдері')
ax.set_xlabel('Нұсқа (test_size)')
ax.set_ylabel('Жол саны')
ax.legend()
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('SOOZh_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def make_split_report(X, y, test_size, random_state):
    """Деректі бөледі және өлшемдер, сынып пайыздары, индекс қиылысын береді."""
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    return {
        'train_rows': len(X_tr),
        'test_rows': len(X_te),
        'train_class_percent': (y_tr.value_counts(normalize=True)*100).round(1).to_dict(),
        'test_class_percent': (y_te.value_counts(normalize=True)*100).round(1).to_dict(),
        'overlap_count': len(set(X_tr.index) & set(X_te.index))
    }

for name, (test_size, seed) in strategies.items():
    print(name, make_split_report(X, y, test_size, seed))

In [ ]:
soo_files = ['SOOZh_comparison.csv', 'SOOZh_comparison.png'] + [
    f'SOOZh_{variant}_{part}.csv'
    for variant in ['A','B','C'] for part in ['train','test']
]
with zipfile.ZipFile('SOOZh_08_results.zip', 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for file_name in soo_files:
        archive.write(file_name)
print('СОӨЖ ZIP жасалды: SOOZh_08_results.zip')

NameError: name 'zipfile' is not defined